In [ ]:
import csv
from collections import defaultdict
import sys

# === PARAMETER CONFIGURATION ===
file_net = "GRN.csv"  # Updated network file
file_logfc = "logFC_patient_x.csv"            # logFC file
file_output = "pruned_net.csv"        # Output file
file_removed = "deleted_links.csv"          # Elimination log file

ACTIVATION_THRESHOLD = 1.0  # Threshold to define a significant gene (|logFC| >= 1)
TAU = 0.3                 # Tolerance zone (Grey Zone)

def carica_logfc(file_path):
    """
    Loads expression data. Returns a dictionary {Gene: logFC}.
    """
    data = {}
    try:
        with open(file_path, "r", encoding='utf-8-sig') as f:
            sample = f.read(1024)
            f.seek(0)
            sniffer = csv.Sniffer()
            try:
                delimiter = sniffer.sniff(sample).delimiter
            except:
                delimiter = ',' # Default fallback
            
            reader = csv.DictReader(f, delimiter=delimiter)
            
            # Remove whitespace from column headers
            reader.fieldnames = [name.strip() for name in reader.fieldnames]
            
            for row in reader:
                # Search for the gene column (handles various common naming conventions)
                gene_key = next((k for k in row.keys() if "Gene" in k or "Symbol" in k), None)
                lfc_key = next((k for k in row.keys() if "logFC" in k or "Value" in k), None)
                
                if gene_key and lfc_key:
                    gene = row[gene_key].strip()
                    try:
                        val = float(row[lfc_key].replace(",", ".")) 
                        data[gene] = val
                    except ValueError:
                        continue
    except Exception as e:
        print(f"Error loading logFC data: {e}")
        sys.exit()
    return data

def check_coerenza(fc_source, fc_target, interaction_type, tau):
    """
    Verifies biological coherence.
    fc_source: Expression value of the SOURCE gene (Source_Gene)
    fc_target: Expression value of the TARGET gene (Target_Gene)
    interaction_type: "+1" (Activation) or "-1" (Inhibition)
    """
    
    # Normalize the interaction type to handle different string formats (e.g., "1", "1.0", "+1")
    # If the file contains exactly "1" or "-1", this conditional check is appropriate.
    # Here, a standard base string comparison is assumed.
    
    is_activation = "1" in interaction_type and "-" not in interaction_type # E.g., "1", "+1"
    is_inhibition = "-" in interaction_type # E.g., "-1"

    # 1. Case: UP-REGULATED SOURCE (S >= 1)
    if fc_source >= 1:
        if is_activation: 
            # If active, the target must not be strongly down-regulated
            return fc_target > -tau
        elif is_inhibition: 
            # If inhibiting, the target must not be strongly up-regulated
            return fc_target < tau

    # 2. Case: DOWN-REGULATED SOURCE (S <= -1) -> Loss of Function
    elif fc_source <= -1:
        if is_activation: # Loss of activation
            # Absence of activator -> Target should not be up-regulated
            return fc_target < tau
        elif is_inhibition: # Loss of inhibition (Release of brake)
            # Absence of inhibitor -> Target is free to increase (must not be down-regulated)
            return fc_target > -tau
            
    # If the source gene expression is not significant, the edge is considered biologically plausible
    # (or at least not contradicted by robust empirical evidence)
    return True

def main():
    print("--- INITIATING DATA PROCESSING ---")
    
    # 1. Data Loading
    logfc_data = carica_logfc(file_logfc)
    print(f"Data successfully loaded: {len(logfc_data)} genes identified.")

    # 2. Identification of Significant Nodes
    significant_genes = {g for g, val in logfc_data.items() if abs(val) >= ACTIVATION_THRESHOLD}
    print(f"Significant Genes (|logFC| >= {ACTIVATION_THRESHOLD}): {len(significant_genes)}")

    edges_kept = []
    edges_removed = []
    
    # 3. Network Reading and Processing
    try:
        with open(file_net, "r", encoding='utf-8-sig') as f:
            sample = f.read(1024)
            f.seek(0)
            sniffer = csv.Sniffer()
            try:
                delimiter = sniffer.sniff(sample).delimiter
            except:
                delimiter = ',' # Default fallback
            
            reader = csv.DictReader(f, delimiter=delimiter)
            reader.fieldnames = [name.strip() for name in reader.fieldnames]

            # Verify the presence of specific required columns
            required_cols = ["Target_Gene", "Source_Gene", "Interaction_Type"]
            if not all(col in reader.fieldnames for col in required_cols):
                print(f"COLUMN ERROR: Identified {reader.fieldnames}. Required columns: {required_cols}")
                return

            for row in reader:
                # --- CORRECT VARIABLE ASSIGNMENT ---
                # Source_Gene represents the upstream regulator (second column)
                # Target_Gene represents the downstream target (first column)
                gene_source = row["Source_Gene"].strip()
                gene_target = row["Target_Gene"].strip()
                tipo = row["Interaction_Type"].strip()

                # TRIGGER CONDITION:
                is_source_sig = gene_source in significant_genes
                is_target_sig = gene_target in significant_genes

                if not is_source_sig and not is_target_sig:
                    # Neither gene is significant -> retain the edge
                    edges_kept.append(row)
                    continue

                # Control for missing data
                if gene_source not in logfc_data or gene_target not in logfc_data:
                    row["Reason"] = "missing_logFC"
                    edges_removed.append(row)
                    continue

                val_source = logfc_data[gene_source]
                val_target = logfc_data[gene_target]

                keep = True
                reason = ""

                # --- VALIDATION ---
                # Always pass (Source, Target) to the check_coerenza function
                
                # CASE A: The source gene is significant (Forward Check)
                if is_source_sig:
                    keep = check_coerenza(val_source, val_target, tipo, TAU)
                    if not keep: 
                        reason = f"Incoherent Outgoing: Source {gene_source} ({val_source}) vs Target {gene_target} ({val_target})"

                # CASE B: The target gene is significant (Backward Check)
                elif is_target_sig: 
                    # Inquiry: "Does the source justify the state of the Target?"
                    keep = check_coerenza(val_source, val_target, tipo, TAU)
                    if not keep: 
                        reason = f"Incoherent Incoming: Source {gene_source} ({val_source}) does not justify Target {gene_target} ({val_target})"
                
                if keep:
                    edges_kept.append(row)
                else:
                    row["Reason"] = reason
                    edges_removed.append(row)

    except Exception as e:
        print(f"Error reading network file: {e}")
        return

    # 4. Result Serialization
    # Maintain the original file column order
    keys_out = ["Target_Gene", "Source_Gene", "Interaction_Type"]
    
    try:
        with open(file_output, "w", newline="", encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=keys_out, delimiter=";", extrasaction='ignore')
            writer.writeheader()
            writer.writerows(edges_kept)

        keys_rem = ["Target_Gene", "Source_Gene", "Interaction_Type", "Reason"]
        with open(file_removed, "w", newline="", encoding='utf-8') as f:
            writer = csv.DictWriter(f, fieldnames=keys_rem, delimiter=";", extrasaction='ignore')
            writer.writeheader()
            writer.writerows(edges_removed)

        print("\n--- FINAL SUMMARY ---")
        print(f"Total edges retained: {len(edges_kept)}")
        print(f"Total edges eliminated: {len(edges_removed)}")
        print(f"Files successfully saved: {file_output}, {file_removed}")
        
    except Exception as e:
        print(f"Error writing output files: {e}")

if __name__ == "__main__":
    main()